# High-Water — a quantitative teardown 🔬
### The nearness hedge with a Lo t-stat · the 0.87 correlation to momentum · decay · costs

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![A distinct anomaly?: Busted](https://img.shields.io/badge/A_distinct_anomaly%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We test whether the 52-week-high effect pays and whether it's distinct from momentum.

> ⚠️ **Not investment advice.** 398 S&P 500 names with ≥20y history (Yahoo), 2000–2026; survivorship-biased, large-cap. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (high_water/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from high_water import data, strategy as st
ret = data.fetch_panel()                       # cache-first; built by examples/verify.py --fetch
hi = st.cross_section_hedge(ret, st.nearness(ret))
mo = st.cross_section_hedge(ret, st.momentum(ret))


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | nearness hedge −8.4%/yr, Sharpe −0.40, t −2.2 |
| Tradability | **Mirage** | negative gross; costs deepen it |
| Distinct anomaly? | **Busted** | 0.87 correlated with momentum |

> 💡 *In plain words:* momentum relabelled, and momentum that lost here.

## 1 · The claim, steelmanned

- **H₁:** the nearness long-short earns a significant positive premium.
- **H₂:** it is distinct from 12-month momentum (low correlation).
- **H₃:** it's stable over time.

## 2 · So what? — what rides on each

If H₁/H₂ hold, it's a new tradable factor. If H₂ fails, it's momentum with a story; if H₁ fails too, there's no premium at all.

## 3 · How we'd know — the protocol

Nearness long-short + Lo t-stat → correlation with the momentum hedge → decade split → cost sweep.

## 4 · The teardown

### 4.1 The nearness hedge vs the momentum control

In [2]:
display(pd.DataFrame({'52-week-high':st.stats(hi),'12m momentum':st.stats(mo)}).T[['mean_ann','sharpe','tstat','hit_rate','n']].round(3))
print(f"corr(nearness hedge, momentum hedge) = {st.signal_overlap(ret):+.2f}")

,mean_ann,sharpe,tstat,hit_rate,n
52-week-high,-0.084,-0.401,-2.206,0.505,366.0
12m momentum,-0.009,-0.044,-0.241,0.540,365.0


corr(nearness hedge, momentum hedge) = +0.87


> 💡 *In plain words:* nearness loses (t −2.2) and is 0.87 correlated with momentum. **H₁ and H₂ both rejected.**

### 4.2 Decay

In [3]:
for lab,sl in [('1999-2012',hi.loc[:'2012']),('2013-on',hi.loc['2013':])]:
    print(f'{lab}: Sharpe {st.stats(sl)["sharpe"]:+.2f}, mean {st.stats(sl)["mean_ann"]:+.2%}/yr')

1999-2012: Sharpe -0.52, mean -13.01%/yr
2013-on: Sharpe -0.18, mean -2.64%/yr


> 💡 *In plain words:* strongly negative early (momentum crashes of 2000/2008), drifting to zero. **H₃** — no stable positive regime.

### 4.3 Cost sweep — a negative book only gets worse

In [4]:
rows={'gross':st.stats(hi)['sharpe']}
for c in (5,10,20): rows[f'{c}bp']=st.stats(st.net_of_cost(hi,c))['sharpe']
display(pd.Series(rows, name='net Sharpe').round(3))

gross   -0.401
5bp     -0.446
10bp    -0.492
20bp    -0.583
Name: net Sharpe, dtype: float64

> 💡 *In plain words:* since gross is negative, turnover just compounds the loss.

## 5 · The verdict

H₁, H₂, H₃ all rejected → Signal `NONE`, Tradability `MIRAGE`, distinct-anomaly claim `BUSTED`.

## 6 · Could you trade it?

No — and if you wanted the momentum exposure it proxies, you'd use momentum directly (cleaner, and itself fragile on large caps — Study 24 Stampede). The 52-week-high label adds a narrative, not an edge.

## 7 · Going further

Forks: (a) a point-in-time / small-cap universe (George-Hwang's effect is stronger in smaller names, the untradable end); (b) double-sort nearness × momentum to confirm there's no residual; (c) the momentum-crash overlay (Daniel-Moskowitz 2016) on the near-high book. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).